# Satellite Damage Assessment: Ukraine Conflict Analysis

## Data Cleaning & Feature Engineering

This notebook processes Sentinel-2 satellite imagery from Microsoft Planetary Computer to detect infrastructure damage across Ukrainian conflict zones (Mariupol, Kharkiv, Bakhmut).

**Pipeline:**
1. Download Sentinel-2 Level-2A from Planetary Computer
2. Apply cloud masking
3. Compute spectral indices (NDVI, NDBI, BSI)
4. Save cleaned dataset for modeling

## 1. Setup & Environment

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt

# Setup paths (notebook runs from ../notebooks/)
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../results/plots', exist_ok=True)

# ============ SPECTRAL INDICES CLASS ============
class SpectralIndices:
    """Compute spectral indices from Sentinel-2 bands."""
    
    @staticmethod
    def ndvi(image):
        """NDVI = (NIR - RED) / (NIR + RED)"""
        red = image[..., 2].astype(float)
        nir = image[..., 3].astype(float)
        return (nir - red) / (nir + red + 1e-8)
    
    @staticmethod
    def ndbi(image):
        """NDBI = (SWIR - NIR) / (SWIR + NIR)"""
        swir = image[..., 4].astype(float)
        nir = image[..., 3].astype(float)
        return (swir - nir) / (swir + nir + 1e-8)
    
    @staticmethod
    def bsi(image):
        """BSI = (SWIR + RED - NIR - BLUE) / (SWIR + RED + NIR + BLUE)"""
        blue = image[..., 0].astype(float)
        red = image[..., 2].astype(float)
        nir = image[..., 3].astype(float)
        swir = image[..., 4].astype(float)
        numerator = swir + red - nir - blue
        denominator = swir + red + nir + blue
        return numerator / (denominator + 1e-8)

# ============ HELPER FUNCTIONS ============
def apply_cloud_mask(image, scl_band):
    """Apply cloud masking using Scene Classification band."""
    valid_pixels = np.isin(scl_band, [4, 5, 6])
    image_masked = image.copy().astype(float)
    image_masked[~valid_pixels] = np.nan
    return image_masked

def normalize_bands(image):
    """Normalize image bands to [0, 1]."""
    return np.clip(image / 10000.0, 0, 1)

def create_patches(image, patch_size=128, stride=64):
    """Extract overlapping patches from image."""
    h, w = image.shape[:2]
    patches = []
    for i in range(0, h - patch_size + 1, stride):
        for j in range(0, w - patch_size + 1, stride):
            patches.append(image[i:i+patch_size, j:j+patch_size])
    return patches

print("✓ Environment ready")

## 2. Download Sentinel-2 Data from Microsoft Planetary Computer

In [ ]:
# Install required packages (run once)
# !pip install pystac-client planetary-computer odc-stac xarray rioxarray

import pystac_client
import planetary_computer
from odc.stac import load

print("Connecting to Microsoft Planetary Computer...")

# Connect to Planetary Computer
catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace
)

print("✓ Connected to Planetary Computer")

# Define study areas
study_areas = {
    'Mariupol': [37.40, 46.95, 37.68, 47.15],  # bbox: lon_min, lat_min, lon_max, lat_max
    'Kharkiv': [35.80, 49.85, 36.50, 50.10],
    'Bakhmut': [37.70, 48.50, 37.90, 48.65]
}

# Time periods
periods = {
    '2021': '2021-06-01/2021-08-31',           # Pre-conflict
    '2022': '2022-08-01/2022-10-31',           # Conflict year 1
    '2023': '2023-06-01/2023-08-31',           # Conflict year 2
    '2024': '2024-05-01/2024-07-31'            # Conflict year 3
}

print(f"\nDownloading Sentinel-2 data...")
print(f"Cities: {list(study_areas.keys())}")
print(f"Time periods: {list(periods.keys())}")
print(f"Expected scenes: {len(study_areas) * len(periods)}\n")

## 3. Download and Process Each Scene

In [ ]:
metadata = []
downloaded_count = 0
failed_count = 0

for city, bbox in study_areas.items():
    for year, datetime_range in periods.items():
        try:
            print(f"Searching {city} {year}...", end=' ')
            
            search = catalog.search(
                collections=['sentinel-2-l2a'],
                bbox=bbox,
                datetime=datetime_range,
                query={'eo:cloud_cover': {'lt': 30}}
            )
            
            items = list(search.get_items())
            if len(items) == 0:
                print("No scenes found")
                continue
            
            item = items[0]
            print(f"Downloading...", end='')
            
            data = load(
                [item],
                bands=['B02', 'B03', 'B04', 'B08', 'B11', 'B12', 'SCL'],
                resolution=20,
                patch_url=planetary_computer.sign_url
            )
            
            # FIX: Extract each band properly
            image_array = []
            for band in ['B02', 'B03', 'B04', 'B08', 'B11', 'B12', 'SCL']:
                band_data = data[band].values
                band_data = np.squeeze(band_data)  # Remove extra dimensions
                image_array.append(band_data)
            
            image = np.stack(image_array, axis=2)  # (H, W, 7 bands)
            
            # Save
            filename = f"../data/raw/S2_{year}_{city}.npy"
            np.save(filename, image)
            
            cloud_cover = item.properties.get('eo:cloud_cover', 0)
            metadata.append({
                'filename': filename,
                'city': city,
                'year': int(year),
                'cloud_cover_pct': cloud_cover,
                'scene_id': item.id
            })
            
            print(f" ✓")
            downloaded_count += 1
            
        except Exception as e:
            print(f" ✗ {str(e)[:40]}")
            failed_count += 1

print(f"\nDownloaded: {downloaded_count} | Failed: {failed_count}")

if metadata:
    df_raw = pd.DataFrame(metadata)
    df_raw.to_csv('../data/raw/metadata.csv', index=False)
    print(df_raw)

## 4. Cloud Masking & Normalization

In [ ]:
# Load metadata
df_raw = pd.read_csv('../data/raw/metadata.csv')

print("Applying cloud mask and normalizing data...\n")

df_cleaned = df_raw.copy()
cleaned_count = 0

for idx, row in df_raw.iterrows():
    try:
        # Load raw scene
        image = np.load(row['filename'])
        
        # Extract spectral bands (0-5) and SCL band (6)
        spectral = image[..., :6].astype(float)
        scl_band = image[..., 6].astype(int)
        
        # Apply cloud mask
        spectral_masked = apply_cloud_mask(spectral, scl_band)
        
        # Normalize to [0, 1]
        spectral_normalized = spectral_masked / 10000.0
        
        # Save cleaned scene
        clean_filename = row['filename'].replace('.npy', '_clean.npy')
        np.save(clean_filename, spectral_normalized)
        
        # Calculate valid pixel ratio
        valid_ratio = np.isfinite(spectral_normalized).sum() / spectral_normalized.size * 100
        df_cleaned.loc[idx, 'valid_pixels_pct'] = valid_ratio
        df_cleaned.loc[idx, 'cleaned_filename'] = clean_filename
        
        cleaned_count += 1
        
    except Exception as e:
        print(f"Error processing {row['filename']}: {e}")
        continue

df_cleaned.to_csv('../data/processed/metadata_cleaned.csv', index=False)

print(f"✓ Cleaned {cleaned_count}/{len(df_raw)} scenes")
print(f"Average valid pixels: {df_cleaned['valid_pixels_pct'].mean():.1f}%")
print(f"\nCleaned metadata saved to: ../data/processed/metadata_cleaned.csv")

## 5. Compute Spectral Indices

In [ ]:
print("Computing spectral indices (NDVI, NDBI, BSI)...\n")

si = SpectralIndices()
indices_data = []

for idx, row in df_cleaned.iterrows():
    try:
        # Load cleaned scene
        image = np.load(row['cleaned_filename'])
        
        # Compute indices
        ndvi = si.ndvi(image)
        ndbi = si.ndbi(image)
        bsi = si.bsi(image)
        
        # Calculate statistics
        indices_data.append({
            'filename': row['cleaned_filename'],
            'city': row['city'],
            'year': row['year'],
            'cloud_cover_pct': row['cloud_cover_pct'],
            'ndvi_mean': np.nanmean(ndvi),
            'ndvi_std': np.nanstd(ndvi),
            'ndvi_min': np.nanmin(ndvi),
            'ndvi_max': np.nanmax(ndvi),
            'ndbi_mean': np.nanmean(ndbi),
            'ndbi_std': np.nanstd(ndbi),
            'bsi_mean': np.nanmean(bsi),
            'bsi_std': np.nanstd(bsi),
            'valid_pixels_pct': row['valid_pixels_pct']
        })
        
    except Exception as e:
        print(f"Error computing indices for {row['cleaned_filename']}: {e}")
        continue

# Create final dataset
df_indices = pd.DataFrame(indices_data)
df_indices.to_csv('../data/processed/sentinel2_clean_2021_2024.csv', index=False)

print(f"✓ Computed indices for {len(indices_data)} scenes")
print(f"\nFinal dataset saved to: ../data/processed/sentinel2_clean_2021_2024.csv")
print(f"\nDataset shape: {df_indices.shape}")
print(f"\nStatistics:")
print(df_indices.describe().round(3))

## 6. Data Quality Summary

In [ ]:
print("DATA QUALITY SUMMARY")
print("="*70)

print(f"\n1. Temporal Coverage:")
for year in sorted(df_indices['year'].unique()):
    count = (df_indices['year'] == year).sum()
    print(f"   {year}: {count} scenes")

print(f"\n2. Geographic Coverage:")
for city in df_indices['city'].unique():
    count = (df_indices['city'] == city).sum()
    print(f"   {city}: {count} scenes")

print(f"\n3. Data Quality Metrics:")
print(f"   Average cloud cover: {df_indices['cloud_cover_pct'].mean():.1f}%")
print(f"   Average valid pixels: {df_indices['valid_pixels_pct'].mean():.1f}%")
print(f"   Missing values: {df_indices.isnull().sum().sum()}")

print(f"\n4. Spectral Index Ranges:")
print(f"   NDVI: [{df_indices['ndvi_mean'].min():.3f}, {df_indices['ndvi_mean'].max():.3f}]")
print(f"   NDBI: [{df_indices['ndbi_mean'].min():.3f}, {df_indices['ndbi_mean'].max():.3f}]")
print(f"   BSI:  [{df_indices['bsi_mean'].min():.3f}, {df_indices['bsi_mean'].max():.3f}]")

print(f"\n" + "="*70)
print("✓ Data cleaning complete!")

## 7. Visualization

In [ ]:
import matplotlib.dates as mdates

# Create year column as x-axis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for city in df_indices['city'].unique():
    city_data = df_indices[df_indices['city'] == city].sort_values('year')
    years = city_data['year'].values
    
    # Plot NDVI
    axes[0, 0].plot(years, city_data['ndvi_mean'], marker='o', label=city, linewidth=2)
    
    # Plot NDBI
    axes[0, 1].plot(years, city_data['ndbi_mean'], marker='s', label=city, linewidth=2)
    
    # Plot BSI
    axes[1, 0].plot(years, city_data['bsi_mean'], marker='^', label=city, linewidth=2)

# Mark conflict start (2022)
for ax in [axes[0, 0], axes[0, 1], axes[1, 0]]:
    ax.axvline(2022, color='red', linestyle='--', alpha=0.7, linewidth=2, label='Conflict Start')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best')

axes[0, 0].set_ylabel('NDVI (Vegetation)', fontsize=11)
axes[0, 0].set_title('Vegetation Index Temporal Evolution', fontsize=12, fontweight='bold')

axes[0, 1].set_ylabel('NDBI (Built-up)', fontsize=11)
axes[0, 1].set_title('Built-up Index Temporal Evolution', fontsize=12, fontweight='bold')

axes[1, 0].set_ylabel('BSI (Bare Soil)', fontsize=11)
axes[1, 0].set_xlabel('Year', fontsize=11)
axes[1, 0].set_title('Bare Soil Index Temporal Evolution', fontsize=12, fontweight='bold')

axes[1, 1].axis('off')

plt.suptitle('Sentinel-2 Spectral Analysis: Ukrainian Conflict Zones', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/spectral_analysis.png', dpi=150, bbox_inches='tight')
print("✓ Visualization saved to: ../results/plots/spectral_analysis.png")
plt.show()

## 8. Summary

In [ ]:
print("\n" + "="*70)
print("FINAL CLEANED DATASET SUMMARY")
print("="*70)
print(df_indices.to_string(index=False))

print(f"\n✓ Main output: ../data/processed/sentinel2_clean_2021_2024.csv")
print(f"✓ Total scenes: {len(df_indices)}")
print(f"✓ Features: {df_indices.shape[1]}")
print(f"\nNext step: Move to Notebook 2 (Feature Engineering)")
print("="*70)